# 03 — Inference

Three modes, all using the same committed `artifacts/model.joblib`:
1. Single prediction from a raw JSON request (like the API does).
2. Batch predictions over the held-out style data.
3. Live call against the FastAPI `/predict` endpoint (Docker/K8s).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import json
import joblib
import pandas as pd
import requests

from src.heartml.data import load_dataset, validate_features
from src.heartml.schema import FEATURES

model = joblib.load(ROOT / "artifacts" / "model.joblib")
metadata = json.loads((ROOT / "artifacts" / "model_metadata.json").read_text())
print("Loaded model:", metadata.get("selected_model", metadata.get("model_name", "unknown")), "| test metrics:", metadata.get("test_metrics", {}))

## 1. Single prediction — sample request

In [ ]:
request = json.loads((ROOT / "sample_request.json").read_text())
features = validate_features(request)      # same validation the API performs
X = pd.DataFrame([features])
proba = model.predict_proba(X)[0, 1]
prediction = int(proba >= 0.5)
print(f"Prediction: {prediction} | Confidence (P(disease)): {proba:.4f}")

## 2. Batch inference over the processed dataset

In [ ]:
df = load_dataset(ROOT / "data" / "processed" / "heart_disease_processed.csv")
X_all = df[FEATURES]
probas = model.predict_proba(X_all)[:, 1]
preds = (probas >= 0.5).astype(int)

out = X_all.copy()
out["prediction"] = preds
out["confidence"] = probas.round(4)
out["actual"] = df["target"]
out["correct"] = out["prediction"] == out["actual"]
out.head(10)

In [ ]:
print(f"Batch accuracy: {out['correct'].mean():.4f} on {len(out)} rows")
(ROOT / "reports").mkdir(parents=True, exist_ok=True)
out.to_csv(ROOT / "reports" / "batch_inference_results.csv", index=False)
print("Saved reports/batch_inference_results.csv")

## 3. Live inference against the deployed API
Start locally with `make api` (or `docker compose up`), then run this cell.
Change BASE_URL to your NodePort / LoadBalancer URL when deployed to K8s.

In [ ]:
BASE_URL = "http://localhost:8000"   # e.g. "http://192.168.49.2:30080" for Minikube NodePort

response = requests.post(
    f"{BASE_URL}/predict", json=request, timeout=10
)
print("Status:", response.status_code)
print("Body:", json.dumps(response.json(), indent=2))

In [ ]:
health = requests.get(f"{BASE_URL}/health", timeout=10)
print("Health:", health.status_code, health.json())

### Notes
- `validate_features` checks required features and converts their values to
  the types expected by the shared preprocessing pipeline.
- Confidence is the model's positive-class probability, **not** a calibrated
  clinical risk (educational prototype only).
- After running section 3, hit `/metrics` and confirm `api_requests_total`
  increased — that's your monitoring evidence.